In [ ]:
# imports

# Auto-reload modules before executing code
%load_ext autoreload
%autoreload 2

import echopype as ep
from pathlib import Path
import numpy as np

from aa_si_calibration import calibration
from aa_si_calibration import standardized_file_lib
from aa_si_calibration import manufacturer_file_parsers

from aa_si_ml import ml
from aa_si_utils import utils
from aa_si_visualization import assorted
from aa_si_calibration import comparison
import pyvista as pv

import xarray as xr

from aa_si_visualization.paraview import sv_dataset_to_vtk

import yaml
import jsonschema

In [ ]:
# User defined variables
# NOTE: must include a .bot file with the same name in the same directory as the raw file!
raw_folder_string = './SWD_20160725-OE61'
raw_file_names = [] # Only use if populated with a list strings of raw file names
netcdf_output_folder_string = './NetCDF-files'
sv_output_folder_string = './Sv-files'
paraview_output_folder_string = './Paraview_files'
paraview_output_file_string = 'nefsc_uc2.vtk'


cal_folder_string = '../calibration_files/HB201607_cal'
output_logs_folder_string = './Output-Logs'
clear_previous_json_logs = True
standardized_calibration_from_report = "HB1603_L1-D20160703_cal_report.yml"
standardized_calibration_from_nc = "HB1603_L1-D20160703_cal_nc_params.yml"
standardized_cal_files_location = "./standardized_cal_files"

standardized_cal_folder = Path(standardized_cal_files_location)
raw_folder = Path(raw_folder_string)
netcdf_output_folder = Path(netcdf_output_folder_string)
netcdf_output_path = netcdf_output_folder / (raw_folder.stem + '.nc') # NOTE: need to calculate this for every raw file if multiple files
sv_output_folder = Path(sv_output_folder_string)
cal_folder = Path(cal_folder_string)
output_logs_folder = Path(output_logs_folder_string)
line_file_path = '../line_files/SpermWhaleClicks_click_data_HB1603_SpermWhaleDive_Span0.2_07252016_2120_UTC.csv'

sv_flag_thresholds = {
                "critical_median": 2.0,
                "large_median": 1.0,
                "moderate_median": 0.5,
                "critical_max": 4.0,
                "large_max": 2.0,
                "moderate_max": 1.0
            }

# Get list of raw files to process
if raw_file_names is not None and len(raw_file_names) > 0:
    # Use specified file names
    raw_file_paths = [raw_folder / filename for filename in raw_file_names]
else:
    # Get all .raw files in folder
    raw_file_paths = sorted(raw_folder.glob("*.raw"))

if not raw_file_paths:
    raise Exception("No raw files found to process")

print(f"Found {len(raw_file_paths)} raw file(s) to process:")
for path in raw_file_paths:
    print(f"  - {path.name}")

if not cal_folder.exists():
    raise Exception("Calibration folder not found")

if not raw_folder.exists():
    raise Exception("Raw folder not found")

# Create output folders if they don't exist
netcdf_output_folder.mkdir(parents=True, exist_ok=True)
sv_output_folder.mkdir(parents=True, exist_ok=True)
output_logs_folder.mkdir(parents=True, exist_ok=True)
standardized_cal_folder.mkdir(parents=True, exist_ok=True)

if clear_previous_json_logs:
    flags_file = Path(output_logs_folder) / "calibration_flags_NEFSC_use_case_1.json"
    if flags_file.exists():
        flags_file.unlink()

In [ ]:
# Open the raw files and convert to netCDF
# Processing Level 1

# Convert each raw file to intermediate netCDF
echodata_nc_list = []
echodata_raw_list = []
netcdf_paths = []
seafloor_depth_list = []  # Store seafloor data separately

for raw_path in raw_file_paths:
    print(f"\nProcessing: {raw_path.name}")

    # Generate netCDF path for this raw file
    netcdf_path = netcdf_output_folder / f"{raw_path.stem}.nc"
    netcdf_paths.append(netcdf_path)

    # Open raw file and convert to netCDF
    ed_elem = ep.open_raw(raw_path, sonar_model="EK60", include_bot=True)
    echodata_raw_list.append(ed_elem)

    # Save to netCDF
    ed_elem.to_netcdf(save_path=netcdf_path)
    print(f"  Converted and saved to: {netcdf_path}")

    # Open converted file and add to list
    echodata_single = ep.open_converted(netcdf_path)
    echodata_nc_list.append(echodata_single)

# Combine all echodata objects if there are multiple files
if len(echodata_nc_list) > 1:
    print(f"\nCombining {len(echodata_nc_list)} echodata objects...")
    echodata = ep.combine_echodata(echodata_nc_list) # For some reason, detected_seafloor_depth is not missing from the nc files in this case

else:
    # Only one file, use it directly
    echodata = echodata_raw_list[0]
    print(f"\nUsing single file")

# Verify combined seafloor data
utils.check_for_seafloor_depth_data(echodata)
print(f"\nFinal echodata ready for processing")

In [ ]:
# Open the raw files and convert to netCDF
# Processing Level 1

# Convert each raw file to intermediate netCDF
echodata_list = []
netcdf_paths = []

for raw_path in raw_file_paths:
    print(f"\n{'='*60}")
    print(f"Processing: {raw_path.name}")
    print(f"{'='*60}")

    # Generate netCDF path for this raw file
    netcdf_path = netcdf_output_folder / f"{raw_path.stem}.nc"
    netcdf_paths.append(netcdf_path)

    # Open raw file and convert to netCDF
    ed_elem = ep.open_raw(raw_path, sonar_model="EK60", include_bot=True)

    # Check what channels are in this file
    print(f"\n📊 Channel information for this file:")
    # Access the beam group dataset properly
    beam_group = ed_elem["Sonar/Beam_group1"]
    channels = beam_group["channel"].values
    freqs = beam_group["frequency_nominal"].values
    print(f"   Number of channels: {len(channels)}")
    for i, (ch, freq) in enumerate(zip(channels, freqs)):
        print(f"   Channel {i}: {ch} - {int(freq/1000)} kHz")

    # Check if backscatter_r has data for each channel
    if 'backscatter_r' in beam_group:
        backscatter = beam_group['backscatter_r']
        print(f"\n   Backscatter_r shape: {backscatter.shape}")
        print(f"   Dimensions: {backscatter.dims}")

        for i, ch in enumerate(channels):
            ch_data = backscatter.sel(channel=ch)
            non_nan_count = (~np.isnan(ch_data.values)).sum()
            total_count = ch_data.size
            percent = (non_nan_count / total_count) * 100
            print(f"   {ch} ({int(freqs[i]/1000)} kHz): {non_nan_count:,} / {total_count:,} ({percent:.1f}%) non-NaN")

    # Save to netCDF
    ed_elem.to_netcdf(save_path=netcdf_path)
    print(f"\n✅ Converted and saved to: {netcdf_path}")

    # Open converted file and add to list
    echodata_single = ep.open_converted(netcdf_path)
    echodata_list.append(echodata_single)

# Combine all echodata objects if there are multiple files
if len(echodata_list) > 1:
    print(f"\n{'='*60}")
    print(f"COMBINING {len(echodata_list)} ECHODATA OBJECTS")
    print(f"{'='*60}")

    echodata = ep.combine_echodata(echodata_list)

    # Check combined channels
    print(f"\n📊 Combined channel information:")
    beam_group_combined = echodata["Sonar/Beam_group1"]
    channels = beam_group_combined["channel"].values
    freqs = beam_group_combined["frequency_nominal"].values
    print(f"   Total channels: {len(channels)}")
    for i, (ch, freq) in enumerate(zip(channels, freqs)):
        print(f"   Channel {i}: {ch} - {int(freq/1000)} kHz")

    # Verify combined seafloor data
    utils.check_for_seafloor_depth_data(echodata)

else:
    # Only one file, use it directly
    echodata = echodata_list[0]
    print(f"\n{'='*60}")
    print(f"Using single file: {netcdf_paths[0]}")
    print(f"{'='*60}")

print(f"\n✅ Final echodata ready for processing")

In [ ]:
# extract calibration parameters from original netcdf file
params = calibration.extract_netcdf_calibration_parameters(echodata, output_logs_folder)

original_cal_params = params["cal_params"]
original_env_params = params["env_params"]
original_other_params = params["other_params"]

# the calibration parameters come from the first file in the sequential raw files that were combined by echopype
nc_cal_files = [str(raw_file_paths[0].name)]

original_other_params["source_filenames_across_channels"] = nc_cal_files
original_other_params["source_file_type"] = ".raw"

In [ ]:
# print parameters from netcdf file
calibration.print_calibration_values(echodata, original_env_params, original_cal_params, original_other_params, "Calibration Values From .nc File")

In [ ]:
report_params = manufacturer_file_parsers.extract_calibration_params_from_EK60_report(
    cal_folder,
    original_other_params["frequency_nominal"],
    output_logs_folder
    )

report_cal_params, report_env_params, report_other_params = manufacturer_file_parsers.convert_ek60_params_to_pipeline_format(report_params)

In [ ]:
# print file calibration parameters
calibration.print_calibration_values(echodata, report_env_params, report_cal_params, report_other_params, "Calibration Values From .cal File Report in .nc format")

In [ ]:
global_params_report = {
    "cruise_id" : "HB1603"
}

file_path_report = standardized_cal_folder / standardized_calibration_from_report

standardized_file_lib.save_cal_params_to_standardized_file(report_cal_params, report_env_params, report_other_params, global_params_report, file_path_report)

In [ ]:
global_params_nc = {
    "cruise_id" : "HB1603"
}

file_path_nc = standardized_cal_folder / standardized_calibration_from_nc

standardized_file_lib.save_cal_params_to_standardized_file(original_cal_params, original_env_params, original_other_params, global_params_nc, standardized_cal_file_path=file_path_nc)

In [ ]:
# Compare the two sets of calibration parameters
comparison.compare_calibration_parameters(report_cal_params, report_env_params, report_other_params, original_cal_params, original_env_params, original_other_params, echodata)

In [ ]:
# Calculate Baseline Sv and Apply seafloor mask:

ds_Sv = ep.calibrate.compute_Sv(echodata)  # obtain a dataset containing Sv, echo_range, and

# Create mask using the corrected function with both raw and calibrated data
mask_blank = utils.createSvMask(ds_Sv)

# Remove seafloor data (and below) with a buffer of 10 m
mask_no_seafloor = utils.remove_seafloor_from_mask(echodata, ds_Sv, mask_blank, buffer_m=100.0)

# remove surface data down to 10 m``
mask = utils.remove_surface_from_mask(ds_Sv, mask_no_seafloor, depth_threshold_m=10.0)

utils.mask_frequency_channels(ds_Sv, mask, [70, 120, 200])

utils.log_mask_stats(mask)

ds_Sv_baseline = utils.apply_mask_to_sv(ds_Sv, mask)

# For testing, skip applying the mask
# ds_Sv_baseline = ds_Sv

# Save the masked dataset
ds_Sv_baseline.to_netcdf(sv_output_folder / "NEFSC_processed_data.nc")
print("Seafloor mask applied and saved")

In [ ]:
# Generate Sv with individual parameters from calibration file
# Processing Level 2

# gain correction
cal_params_gain_only = {'gain_correction': report_cal_params["gain_correction"]}
ds_Sv_gain_test_unmasked = ep.calibrate.compute_Sv(echodata, cal_params=cal_params_gain_only)
ds_Sv_gain_test = utils.apply_mask_to_sv(ds_Sv_gain_test_unmasked, mask)

# sa correction
cal_params_sa_only = {'sa_correction': report_cal_params["sa_correction"]}
ds_Sv_sa_test_unmasked = ep.calibrate.compute_Sv(echodata, cal_params=cal_params_sa_only)
ds_Sv_sa_test = utils.apply_mask_to_sv(ds_Sv_sa_test_unmasked, mask)

# equivalent beam angle
cal_params_eba_only = {'equivalent_beam_angle': report_cal_params["equivalent_beam_angle"]}
ds_Sv_eba_test_unmasked = ep.calibrate.compute_Sv(echodata, cal_params=cal_params_eba_only)
ds_Sv_eba_test = utils.apply_mask_to_sv(ds_Sv_eba_test_unmasked, mask)

# equivalent beam angle
env_params_ss_only = {'sound_speed': report_env_params["sound_speed"]}
ds_Sv_ss_test_unmasked = ep.calibrate.compute_Sv(echodata, env_params=env_params_ss_only)
ds_Sv_sound_speed_test = utils.apply_mask_to_sv(ds_Sv_ss_test_unmasked, mask)

# absorption
env_params_ab_only = {'sound_absorption': report_env_params["sound_absorption"]}
ds_Sv_ab_test_unmasked = ep.calibrate.compute_Sv(echodata, env_params=env_params_ab_only)
ds_Sv_absorption_test = utils.apply_mask_to_sv(ds_Sv_ab_test_unmasked, mask)

# Generate Sv with all parameters from calibration file combined
cal_params_combined = {
    'gain_correction': report_cal_params["gain_correction"],
    'sa_correction': report_cal_params["sa_correction"],
    'equivalent_beam_angle': report_cal_params["equivalent_beam_angle"]
    }

env_params_combined = {
    'sound_speed': report_env_params["sound_speed"],
    'sound_absorption': report_env_params["sound_absorption"]
    }

ds_Sv_combined_test_unmasked = ep.calibrate.compute_Sv(echodata, cal_params=cal_params_combined, env_params=env_params_combined)
ds_Sv_combined_test = utils.apply_mask_to_sv(ds_Sv_combined_test_unmasked, mask)

In [ ]:
# calculate and print the effect of applying each calibration parameter as well as the combined effect

gain_results = comparison.calculate_full_dataset_effect(ds_Sv_gain_test, ds_Sv_baseline, "Gain Correction", output_logs_folder, sv_flag_thresholds)

sa_results = comparison.calculate_full_dataset_effect(ds_Sv_sa_test, ds_Sv_baseline, "Sa Correction", output_logs_folder, sv_flag_thresholds)

eba_results = comparison.calculate_full_dataset_effect(ds_Sv_eba_test, ds_Sv_baseline, "Equivalent Beam Angle", output_logs_folder, sv_flag_thresholds)

sound_speed_results = comparison.calculate_full_dataset_effect(ds_Sv_sound_speed_test, ds_Sv_baseline, "Sound Speed", output_logs_folder, sv_flag_thresholds)

absorption_results = comparison.calculate_full_dataset_effect(ds_Sv_absorption_test, ds_Sv_baseline, "Absorption", output_logs_folder, sv_flag_thresholds)

combined_results = comparison.calculate_full_dataset_effect(ds_Sv_combined_test, ds_Sv_baseline, "Combined Results", output_logs_folder, sv_flag_thresholds)

In [ ]:
# plot effects of each calibration parameter

comparison.sv_difference_summary_stats_plot(
    ds_Sv_baseline,
    ds_Sv_absorption_test,
    "Absorption Differences"
)

comparison.sv_difference_summary_stats_plot(
    ds_Sv_baseline,
    ds_Sv_gain_test,
    "Gain Differences"
)

comparison.sv_difference_summary_stats_plot(
    ds_Sv_baseline,
    ds_Sv_sa_test,
    "Sa Correction Differences"
)

comparison.sv_difference_summary_stats_plot(
    ds_Sv_baseline,
    ds_Sv_sound_speed_test,
    "Sound Speed Differences"
)

comparison.sv_difference_summary_stats_plot(
    ds_Sv_baseline,
    ds_Sv_combined_test,
    "Combined Differences"
)

In [ ]:
# Perform range-dependent analysis with Absorption results
comparison.perform_range_analysis(ds_Sv_baseline, ds_Sv_absorption_test, echodata, "Absorption Effect Range Analysis")

In [ ]:
# VISUALIZATION OF CALIBRATION DIFFERENCES

min_depth = 0
max_depth = 1200
ping_min = 000
ping_max = 1000

assorted.sv_differences_echograms(ds_Sv_baseline, ds_Sv_combined_test, original_other_params["frequency_nominal"], max_depth, min_depth, ping_min, ping_max, x_axis_units="pings", y_axis_units="meters")

In [ ]:
# Verify that individual parameter effects add up to the combined effect
verification_results = comparison.verify_additive_effects(
    gain_results,
    sa_results,
    eba_results,
    sound_speed_results,
    absorption_results,
    combined_results
)

In [ ]:
# Ensure pvxarray accessor is registered
try:
    import pvxarray.accessor
    from pvxarray.accessor import PyVistaAccessor
    xr.register_dataset_accessor("pyvista")(PyVistaAccessor)
    print(" PyVista accessor registered")
except Exception as e:
    print(f" PyVista accessor registration failed: {e}")

    # For Paraview: export Sv dataset as a VTK file with all channels
output_file = Path(paraview_output_folder_string) / paraview_output_file_string
grid = sv_dataset_to_vtk(ds_Sv_combined_test, output_file)

In [ ]:
line_name = "sw_dive_profile"
ds_Sv_clean, ds_MVBS = ml.data_preprocessing_pipeline(
    ds_Sv_baseline,
    echodata,
    noise_range_sample_num=10,
    noise_ping_num=5,
    mvbs_range_bin="2m",
    mvbs_ping_time_bin="10s",
    mvbs_nan_threshold=.6,
    plot_window=[0, 1800, 0, 515],
    overlay_line_var=line_name,
    overlay_line_path=line_file_path
    )

In [ ]:
cluster_colors = [
                "#FFAD09", "#35E200", "#0700DC", "#EF0000", "#DC0303",
                "#FFFB1C", "#4E9200", "#970021", "#8E00E0", "#017685FF", "#4100B9FF"
            ]
dataset_name = "ml_dataset_6"
custom_normalization_name = "normalized_data_flatten_mean_centered_6"
ml_result_name = "dbscan_background_results_6"

ds_MVBS, gridded_background_results_dbscan, dbscan_results = ml.full_dbscan_iteration(
    ds_MVBS,
    dataset_name,
    ds_Sv_clean,
    feature_strategy="mean_centered", # CLR
    data_var="Sv",
    custom_normalization_name=custom_normalization_name,
    ml_result_name=ml_result_name,
    normalization_strategy="flatten", # Yeo Johnson + PIT
    feature_weights=[1, 1, 0],
    plot_window=[0, 1800, 0, 515],
    min_samples=10, # usually ~ 2-10
    sample_size=1000000, # use full dataset if smaller
    min_cluster_size=1400, # usually ~ 1000 - 2000 for larger datasets
    cluster_selection_method="eom", # also worth trying leaf
    use_hdbscan=True,
    baseline_channel=0,
    soft_membership_threshold=.4,
    cluster_colors=cluster_colors,
    overlay_line_var=line_name
    )
